# QBE: data quality rules with DQX

Runs the QBE data quality rule catalog with [Databricks Labs DQX](https://databrickslabs.github.io/dqx/) and
persists the results: a pass rate per rule compared with its thresholds, and every failing record.

**Config** (change these, not the notebook; paths are relative to this notebook's folder):

| File | What it holds |
|---|---|
| `config/tables/<table>.yml` | One file per table: the catalog's source table, the table actually read, and its rules grouped by column (rule_id, business term, dimension, thresholds, and the DQX check). Rules over several columns go under `table_rules` |
| `config/qbe_validation.yml` | Where the table files are, where results are written, and run options |

**Results** (each can be switched off and set to `append` or `overwrite` in `qbe_validation.yml`):

| Table | Contents |
|---|---|
| `qbe_dq_rule_summary` | One row per rule per run: rows checked, rows in scope, rows failed, pass rate, minimum/target threshold, `threshold_status` (`met`, `below_target`, `below_minimum`, `no_rows`, `error`) |
| `qbe_dq_failed_records` | One row per failing record per rule: `rule_id`, `rule_name`, `criticality`, `message`, and the record as JSON |

**Parameters:** every entry under `variables` in `qbe_validation.yml` is a notebook parameter with that default,
used for `${name}` in the config: `source_catalog` (the tables read) and `results_catalog` (the results tables). A job passes values with `base_parameters`;
for a manual run, change them in the widgets at the top. `dq_job_run_id` is the job run id when run by a job,
`manual` otherwise. The notebook only needs DQX.

In [ ]:
%pip install databricks-labs-dqx==0.16.0

In [ ]:
%restart_python

In [ ]:
import json


def current_job_run_id() -> str:
    """The job run id when this notebook runs as a job task; "manual" for interactive runs."""
    try:
        context = json.loads(dbutils.notebook.entry_point.getDbutils().notebook().getContext().safeToJson())
        attributes = context.get("attributes", {})
        return str(attributes.get("multitaskParentRunId") or attributes.get("jobRunId") or "manual")
    except Exception:
        return "manual"


job_run_id = current_job_run_id()

In [ ]:
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path

import yaml
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

# Config paths are relative to this notebook's folder (use_cases/qbe)
SRC_DIR = Path.cwd()


def read_yaml(relative_path: str) -> dict:
    with open(SRC_DIR / relative_path) as yaml_file:
        return yaml.safe_load(yaml_file)


def rules_of(table_config: dict) -> list:
    """A table file's enabled rules, each with its column, business term and source table filled in."""
    rules = []
    for column, column_config in (table_config.get("columns") or {}).items():
        for rule in column_config.get("rules") or []:
            rules.append({"business_term": column_config.get("business_term"), "cde_id": column_config.get("cde_id"),
                          **rule, "column": column, "source_table": table_config["source_table"]})
    for rule in table_config.get("table_rules") or []:
        rules.append({**rule, "column": None, "source_table": table_config["source_table"]})
    return [rule for rule in rules if rule.get("enabled", True)]


def parameter(name: str, default: str) -> str:
    """A notebook parameter (widget) with a default; a job sets it with base_parameters."""
    try:
        dbutils.widgets.text(name, default)
        return dbutils.widgets.get(name)
    except NameError:               # no dbutils outside Databricks: use the default
        return default


def resolve(value, variables: dict):
    """Replaces ${name} in every string of a config with the variable's value; unknown names fail."""
    if isinstance(value, str):
        def lookup(match):
            if match.group(1) not in variables:
                raise ValueError(f"${{{match.group(1)}}} is not defined under variables in qbe_validation.yml")
            return variables[match.group(1)]
        return re.sub(r"\$\{([^}]+)\}", lookup, value)
    if isinstance(value, dict):
        return {key: resolve(item, variables) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve(item, variables) for item in value]
    return value


raw_settings = read_yaml("config/qbe_validation.yml")["qbe_validation"]
variables = {name: parameter(name, str(default)) for name, default in (raw_settings.get("variables") or {}).items()}
settings = resolve(raw_settings, variables)
table_configs = [resolve(read_yaml(path.relative_to(SRC_DIR)), variables)
                 for path in sorted((SRC_DIR / settings["tables_dir"]).glob("*.yml"))]
rules_by_table = {table_config["source_table"]: rules_of(table_config) for table_config in table_configs}

rule_ids = [rule["rule_id"] for table_rules in rules_by_table.values() for rule in table_rules]
duplicates = sorted({rule_id for rule_id in rule_ids if rule_ids.count(rule_id) > 1})
if duplicates:
    raise ValueError(f"rule_id used more than once across {settings['tables_dir']}: {', '.join(duplicates)}")

RESULT_MODES = ("append", "overwrite")
for result_name, result in settings["results"].items():
    if result.get("mode", "append") not in RESULT_MODES:
        raise ValueError(f"results.{result_name}.mode must be one of {RESULT_MODES}, got: {result.get('mode')}")

dq_engine = DQEngine(WorkspaceClient())
run_id = str(uuid.uuid4())
run_ts = datetime.now(timezone.utc).replace(tzinfo=None)

print(f"run_id={run_id} job_run_id={job_run_id} variables={variables} tables={len(table_configs)} rules={len(rule_ids)}")
for result_name, result in settings["results"].items():
    print(f"{result_name}: " + (f"{result['table']} ({result.get('mode', 'append')})" if result.get("enabled", True) else "not persisted"))

In [ ]:
SUMMARY_SCHEMA = (
    "dq_run_id string, dq_run_ts timestamp, dq_job_run_id string, rule_id string, rule_name string, "
    "business_term string, dq_dimension string, source_table string, physical_table string, column_name string, "
    "criticality string, rows_checked long, rows_in_scope long, rows_failed long, pass_rate double, "
    "minimum_threshold double, target_threshold double, threshold_status string, error string"
)


def to_dqx_check(rule: dict) -> dict:
    """The DQX check for a rule: named by rule_id, on its column (column rules), filtered by its boundary conditions."""
    function = rule["check"]["function"]
    arguments = dict(rule["check"].get("arguments") or {})
    if rule["column"] and function == "sql_expression":
        arguments.setdefault("columns", [rule["column"]])
    elif rule["column"]:
        arguments.setdefault("column", rule["column"])
    check = {"name": rule["rule_id"], "criticality": rule.get("criticality", "error"),
             "check": {"function": function, "arguments": arguments}}
    if rule.get("boundary_conditions"):
        check["filter"] = rule["boundary_conditions"]
    return check


def read_source(table_config: dict):
    """Reads a table file's table, with physical column names renamed to the catalog's column names."""
    df = spark.read.table(table_config["table"])
    renames = {column: column_config["physical_column"] for column, column_config in (table_config.get("columns") or {}).items()
               if column_config.get("physical_column")}
    missing = [physical for physical in renames.values() if physical not in df.columns]
    if missing:
        raise ValueError(f"columns not in {table_config['table']}: {', '.join(missing)}")
    return df.withColumnsRenamed({physical: column for column, physical in renames.items()})


def threshold_status(pass_rate, rule: dict) -> str:
    if pass_rate is None:
        return "no_rows"
    minimum, target = rule.get("minimum_threshold"), rule.get("target_threshold")
    if minimum is not None and pass_rate < minimum:
        return "below_minimum"
    if target is not None and pass_rate < target:
        return "below_target"
    return "met"


def summary_row(rule: dict, table_config: dict, **values) -> dict:
    as_float = lambda value: float(value) if value is not None else None
    return {
        "dq_run_id": run_id, "dq_run_ts": run_ts, "dq_job_run_id": job_run_id,
        "rule_id": rule["rule_id"], "rule_name": rule.get("rule_name"), "business_term": rule.get("business_term"),
        "dq_dimension": rule.get("dq_dimension"), "source_table": table_config["source_table"],
        "physical_table": table_config["table"],
        "column_name": rule["column"] or ", ".join(rule["check"].get("arguments", {}).get("columns") or []),
        "criticality": rule.get("criticality", "error"),
        "minimum_threshold": as_float(rule.get("minimum_threshold")), "target_threshold": as_float(rule.get("target_threshold")),
        "rows_checked": None, "rows_in_scope": None, "rows_failed": None, "pass_rate": None,
        "threshold_status": None, "error": None, **values,
    }


def is_rule(rule_id: str):
    """A one-argument predicate for F.exists: whether a DQX result belongs to the rule."""
    return lambda result: result["name"] == rule_id


def validate_table(table_config: dict, table_rules: list) -> tuple:
    """Runs a table's rules in one DQX pass; returns one summary row per rule and the failing records."""
    checks = [to_dqx_check(rule) for rule in table_rules]
    status = DQEngine.validate_checks(checks)
    if status.has_errors:
        raise ValueError(f"Invalid DQX checks for {table_config['source_table']}: {status}")

    df = read_source(table_config)
    checked = dq_engine.apply_checks_by_metadata(df, checks)
    if isinstance(checked, tuple):      # DQX adds an Observation when an observer is configured
        checked = checked[0]

    # One pass over the data: rows checked, and per rule the rows in scope and the rows that failed it
    aggregations = [F.count("*").alias("rows_checked")]
    for index, rule in enumerate(table_rules):
        in_scope = F.expr(rule["boundary_conditions"]) if rule.get("boundary_conditions") else F.lit(True)
        failed = F.exists("_errors", is_rule(rule["rule_id"])) | F.exists("_warnings", is_rule(rule["rule_id"]))
        aggregations += [F.count(F.when(in_scope, 1)).alias(f"in_scope_{index}"), F.count(F.when(failed, 1)).alias(f"failed_{index}")]
    counts = checked.agg(*aggregations).first()

    rows = []
    for index, rule in enumerate(table_rules):
        in_scope, failed = counts[f"in_scope_{index}"], counts[f"failed_{index}"]
        pass_rate = round((in_scope - failed) / in_scope * 100, 4) if in_scope else None
        rows.append(summary_row(rule, table_config, rows_checked=counts["rows_checked"], rows_in_scope=in_scope,
                                rows_failed=failed, pass_rate=pass_rate, threshold_status=threshold_status(pass_rate, rule)))
    return rows, failures_of(checked, table_config, table_rules)


def failures_of(checked, table_config: dict, table_rules: list):
    """One row per failing record per rule, with the rule and the record (record_columns, or the full row) as JSON."""
    source_columns = [c for c in checked.columns if c not in ("_errors", "_warnings")]
    record_columns = table_config.get("record_columns") or source_columns
    missing = [c for c in record_columns if c not in source_columns]
    if missing:
        raise ValueError(f"record_columns not in {table_config['source_table']}: {', '.join(missing)}")
    record = F.to_json(F.struct(*[F.col(f"`{c}`") for c in record_columns]))
    rule_names = F.create_map(*[F.lit(v) for rule in table_rules for v in (rule["rule_id"], rule.get("rule_name") or "")])
    results = (
        checked.select(record.alias("record"), F.explode("_errors").alias("result"), F.lit("error").alias("criticality"))
        .unionByName(checked.select(record.alias("record"), F.explode("_warnings").alias("result"), F.lit("warn").alias("criticality")))
    )
    return results.select(
        F.lit(run_id).alias("dq_run_id"),
        F.lit(run_ts).alias("dq_run_ts"),
        F.lit(job_run_id).alias("dq_job_run_id"),
        F.lit(table_config["source_table"]).alias("source_table"),
        F.lit(table_config["table"]).alias("physical_table"),
        F.col("result.name").alias("rule_id"),
        F.element_at(rule_names, F.col("result.name")).alias("rule_name"),
        "criticality",
        F.col("result.message").alias("message"),
        F.col("result.columns").alias("columns"),
        "record",
    )


def persist(result_name: str, df) -> None:
    """Writes one result (rule_summary or failed_records) as configured under results.<result_name>."""
    result = settings["results"][result_name]
    if not result.get("enabled", True):
        return
    mode = result.get("mode", "append")
    writer = df.write.mode(mode)
    writer = writer.option("overwriteSchema", "true") if mode == "overwrite" else writer.option("mergeSchema", "true")
    writer.saveAsTable(result["table"])

In [ ]:
summary_rows, failures_dfs = [], []
for table_config in table_configs:
    table_rules = rules_by_table[table_config["source_table"]]
    if not table_rules:
        continue
    try:
        rows, failures = validate_table(table_config, table_rules)
        summary_rows += rows
        failures_dfs.append(failures)
    except Exception as error:
        summary_rows += [summary_row(rule, table_config, threshold_status="error", error=str(error)[:2000]) for rule in table_rules]

# Each result is written once per run, so "overwrite" keeps the whole run, not just the last table
if failures_dfs:
    all_failures = failures_dfs[0]
    for failures in failures_dfs[1:]:
        all_failures = all_failures.unionByName(failures)
    persist("failed_records", all_failures)

summary = spark.createDataFrame(summary_rows, SUMMARY_SCHEMA)
persist("rule_summary", summary)
display(summary)

In [ ]:
errored = sorted({row["source_table"] for row in summary_rows if row["threshold_status"] == "error"})
if errored:
    raise RuntimeError(f"Rules could not run for: {', '.join(errored)}. See the error column in the results.")

breached = [row["rule_id"] for row in summary_rows
            if row["threshold_status"] in ("below_target", "below_minimum") and row["criticality"] == "error"]
if breached and settings.get("fail_on_threshold_breach", False):
    raise RuntimeError(f"Rules below their target threshold: {', '.join(breached)}")